In [1]:
import cv2
import numpy as np
import mediapipe as mp
import time
import json
import math
import pyrealsense2 as rs  # Intel RealSense SDK

SCAN_DURATION = 5.0   # 스캔 시간(초) ≈ 4~5초
SAFE_RADIUS   = 50.0  # 스캔 후 얼굴 중심이 이 픽셀 거리 이상 벗어나면 경고

# 얼굴 외곽(얼굴 타원)로 자주 쓰이는 FaceMesh 인덱스들 (대략적인 oval)
FACE_OVAL_IDX = [
    10, 338, 297, 332, 284, 251, 389, 356,
    454, 323, 361, 288, 397, 365, 379, 378,
    400, 377, 152, 148, 176, 149, 150, 136,
    172, 58, 132, 93, 234, 127, 162, 21,
    54, 103, 67, 109
]

# 왼/오 볼 윤곽을 이루는 랜드마크 인덱스 (기존 그대로 유지: 참조/시각화용)
LEFT_CHEEK_IDX  = [234,  93, 132,  58, 172, 136, 150, 176, 148, 152]
RIGHT_CHEEK_IDX = [454, 323, 361, 288, 397, 365, 379, 400, 377, 152]

# 눈/코/입 근처를 대략적으로 정의하기 위한 랜드마크 몇 개
LEFT_EYE_CENTER_IDX  = [33, 133]   # 왼쪽 눈 근처 두 점 평균
RIGHT_EYE_CENTER_IDX = [362, 263]  # 오른쪽 눈 근처 두 점 평균
NOSE_CENTER_IDX      = [1, 4]      # 코 중앙/끝 근처
MOUTH_CENTER_IDX     = [13, 14]    # 입 윗/아랫 부분 근처

# 눈/코/입 주변을 제외하기 위한 픽셀 반경 (대략)
EXCLUDE_EYE_RADIUS   = 35
EXCLUDE_NOSE_RADIUS  = 30
EXCLUDE_MOUTH_RADIUS = 40


def get_point_mean(landmarks, idx_list, w, h):
    """landmarks에서 idx_list에 해당하는 좌표들 평균 (u, v) 리턴"""
    pts = []
    for idx in idx_list:
        lm = landmarks.landmark[idx]
        u = int(lm.x * w)
        v = int(lm.y * h)
        pts.append((u, v))
    if not pts:
        return None
    mx = int(sum(p[0] for p in pts) / len(pts))
    my = int(sum(p[1] for p in pts) / len(pts))
    return mx, my


def main():
    # ==============
    # 1. RealSense 초기화
    # ==============
    pipeline = rs.pipeline()
    config = rs.config()

    # 컬러 + 뎁스 스트림 설정
    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
    config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)

    profile = pipeline.start(config)

    # ✅ 컬러 카메라 intrinsics (3D 변환용)
    color_profile = profile.get_stream(rs.stream.color)
    intr = color_profile.as_video_stream_profile().get_intrinsics()
    fx, fy, cx, cy = intr.fx, intr.fy, intr.ppx, intr.ppy
    print(f"[INFO] Color intrinsics: fx={fx:.2f}, fy={fy:.2f}, cx={cx:.2f}, cy={cy:.2f}")

    # depth를 color에 align
    align_to = rs.stream.color
    align = rs.align(align_to)

    # ==============
    # 2. MediaPipe FaceMesh 초기화
    # ==============
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    # 볼 영역 bounding box 확장 margin (픽셀 단위) – 지금은 시각화용
    margin_x = 15
    margin_y = 15

    # ==============
    # 스캔/안전영역 관리용 변수
    # ==============
    scan_start_time = time.time()
    scanning = True  # True: 얼굴 영역 좌표 스캔 모드, False: 안전영역 모니터링 모드

    scan_targets = []  # 스캔 동안 얼굴 영역의 좌표들 저장용
    safe_center = None  # 스캔 종료 시 얼굴 기준 중심
    was_inside_safe = True  # 바로 직전 프레임에서 안전영역 안이었는지

    try:
        while True:
            now = time.time()
            elapsed = now - scan_start_time

            # 스캔 시간 종료 체크
            if scanning and elapsed >= SCAN_DURATION:
                scanning = False
                print(f"[INFO] 스캔 종료 ({elapsed:.2f}초). 이제 안전 영역만 모니터링합니다.")
                # safe_center는 아래 FaceMesh 처리에서 최신 얼굴 중심으로 설정될 것

            # ==============
            # 3. D435에서 프레임 받기 (color + depth aligned)
            # ==============
            frames = pipeline.wait_for_frames()
            aligned_frames = align.process(frames)

            color_frame = aligned_frames.get_color_frame()
            depth_frame = aligned_frames.get_depth_frame()
            if not color_frame or not depth_frame:
                continue

            frame = np.asanyarray(color_frame.get_data())  # BGR
            h, w, _ = frame.shape

            # ==============
            # 4. FaceMesh로 얼굴 polygon + 볼/얼굴 중심 계산
            # ==============
            rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb_image)

            left_poly = []
            right_poly = []
            face_oval_poly = []
            face_center = None  # 현재 프레임 기준 얼굴 중심 (볼 기준)

            # 눈/코/입 중심
            left_eye_center = None
            right_eye_center = None
            nose_center = None
            mouth_center = None

            if results.multi_face_landmarks:
                face_landmarks = results.multi_face_landmarks[0]

                # --- 얼굴 외곽(oval) polygon ---
                for idx in FACE_OVAL_IDX:
                    lm = face_landmarks.landmark[idx]
                    u = int(lm.x * w)
                    v = int(lm.y * h)
                    face_oval_poly.append((u, v))
                if len(face_oval_poly) >= 3:
                    oval_np = np.array(face_oval_poly, np.int32)
                    cv2.polylines(frame, [oval_np], True, (0, 255, 0), 1)

                # --- 왼쪽 볼 (시각화/center 계산용) ---
                for idx in LEFT_CHEEK_IDX:
                    lm = face_landmarks.landmark[idx]
                    u = int(lm.x * w)
                    v = int(lm.y * h)
                    left_poly.append((u, v))
                    cv2.circle(frame, (u, v), 2, (0, 0, 255), -1)  # 빨강 점

                # --- 오른쪽 볼 (시각화/center 계산용) ---
                for idx in RIGHT_CHEEK_IDX:
                    lm = face_landmarks.landmark[idx]
                    u = int(lm.x * w)
                    v = int(lm.y * h)
                    right_poly.append((u, v))
                    cv2.circle(frame, (u, v), 2, (255, 0, 0), -1)  # 파랑 점

                # 얼굴 중심(볼들 평균 위치) 계산
                cheek_points = left_poly + right_poly
                if cheek_points:
                    cx_mean = int(sum(p[0] for p in cheek_points) / len(cheek_points))
                    cy_mean = int(sum(p[1] for p in cheek_points) / len(cheek_points))
                    face_center = (cx_mean, cy_mean)
                    cv2.circle(frame, face_center, 4, (0, 255, 255), -1)  # 노랑

                    # 스캔이 끝난 뒤, 첫 얼굴 중심을 안전 기준으로 설정
                    if not scanning and safe_center is None:
                        safe_center = face_center
                        print(f"[INFO] 안전 영역 기준 중심 설정: {safe_center}")

                # 눈/코/입 중심 대략 계산
                left_eye_center  = get_point_mean(face_landmarks, LEFT_EYE_CENTER_IDX,  w, h)
                right_eye_center = get_point_mean(face_landmarks, RIGHT_EYE_CENTER_IDX, w, h)
                nose_center      = get_point_mean(face_landmarks, NOSE_CENTER_IDX,      w, h)
                mouth_center     = get_point_mean(face_landmarks, MOUTH_CENTER_IDX,     w, h)

                # 시각화 (원하면 주석 해제)
                for c, col in [
                    (left_eye_center,  (0, 255, 255)),
                    (right_eye_center, (0, 255, 255)),
                    (nose_center,      (0, 165, 255)),
                    (mouth_center,     (255, 255, 0)),
                ]:
                    if c is not None:
                        cv2.circle(frame, c, 3, col, -1)

            # ==============
            # 5. 스캔 모드일 때: 얼굴 외곽 mask 만들고,
            #    눈/코/입 주변은 제외한 영역의 모든 좌표 + depth + 3D 수집
            # ==============
            if scanning and face_oval_poly:
                face_mask = np.zeros((h, w), dtype=np.uint8)

                # 1) 얼굴 외곽(oval) 채우기
                cv2.fillPoly(face_mask, [np.array(face_oval_poly, np.int32)], 255)

                # 2) 눈/코/입 주변 영역을 0으로 만들어서 제외
                def erase_circle(center, radius):
                    if center is None:
                        return
                    cx, cy = center
                    cv2.circle(face_mask, (cx, cy), radius, 0, -1)

                erase_circle(left_eye_center,  EXCLUDE_EYE_RADIUS)
                erase_circle(right_eye_center, EXCLUDE_EYE_RADIUS)
                erase_circle(nose_center,      EXCLUDE_NOSE_RADIUS)
                erase_circle(mouth_center,     EXCLUDE_MOUTH_RADIUS)

                ys, xs = np.where(face_mask == 255)

                points_this_frame = []
                for v, u in zip(ys, xs):
                    depth_m = float(depth_frame.get_distance(int(u), int(v)))
                    if depth_m <= 0:
                        continue

                    # 카메라 좌표계 기준 3D
                    X = (u - cx) / fx * depth_m
                    Y = (v - cy) / fy * depth_m
                    Z = depth_m

                    points_this_frame.append({
                        "u": int(u),
                        "v": int(v),
                        "depth_m": depth_m,
                        "X_m": X,
                        "Y_m": Y,
                        "Z_m": Z,
                    })

                    # 모든 유효점 시각화
                    cv2.circle(frame, (int(u), int(v)), 1, (0, 255, 0), -1)

                scan_targets.append({
                    "timestamp": now,
                    "points": points_this_frame,
                })



            # ==============
            # 6. 스캔 이후: 얼굴 안전 영역 모니터링
            # ==============
            if not scanning and safe_center is not None and face_center is not None:
                dx = face_center[0] - safe_center[0]
                dy = face_center[1] - safe_center[1]
                dist = math.hypot(dx, dy)

                # 안전 영역 시각화 (원)
                cv2.circle(frame, safe_center, int(SAFE_RADIUS), (0, 255, 0), 1)

                now_inside_safe = dist <= SAFE_RADIUS

                # 안에 있다가 처음 밖으로 나가는 순간에만 경고 출력
                if was_inside_safe and not now_inside_safe:
                    print("얼굴이 안전 영역을 벗어났습니다. 정지합니다.")

                was_inside_safe = now_inside_safe

            # ==============
            # 7. 화면 표시
            # ==============
            if scanning:
                cv2.putText(
                    frame,
                    f"SCANNING (face skin area)... {elapsed:.1f}s / {SCAN_DURATION:.1f}s",
                    (10, 20),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 255, 255),
                    2,
                    cv2.LINE_AA,
                )
            else:
                cv2.putText(
                    frame,
                    "SAFE MONITORING",
                    (10, 20),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 255, 255),
                    2,
                    cv2.LINE_AA,
                )

            cv2.imshow("D435 + Face Skin Region (All Points) + Safe Zone", frame)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break

    finally:
        # 스캔 결과를 JSON 파일로 저장
        try:
            with open("scan_targets.json", "w", encoding="utf-8") as f:
                json.dump(scan_targets, f, ensure_ascii=False, indent=2)
            print(f"[INFO] 스캔 타겟 프레임 {len(scan_targets)}개를 scan_targets.json에 저장했습니다.")
        except Exception as e:
            print(f"[WARN] 스캔 결과 저장 중 오류: {e}")

        face_mesh.close()
        pipeline.stop()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    main()


[INFO] Color intrinsics: fx=607.05, fy=606.54, cx=328.83, cy=243.30


c:\Users\park9\anaconda3\envs\Pproject\lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


[INFO] 스캔 종료 (5.03초). 이제 안전 영역만 모니터링합니다.
[INFO] 안전 영역 기준 중심 설정: (416, 328)
얼굴이 안전 영역을 벗어났습니다. 정지합니다.
[INFO] 스캔 타겟 프레임 36개를 scan_targets.json에 저장했습니다.


KeyboardInterrupt: 